# Music Genre Classification
1. Proper Train/Val/Test split (no data leakage)
2. Combined `features_3_sec` + `features_30_sec` datasets
3. Wider architecture (512â†’256â†’128â†’64â†’10) with L2 regularization
4. EarlyStopping + ModelCheckpoint callbacks
5. Label Smoothing in loss function

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.regularizers import l2
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading & Combining Both Datasets

In [2]:
# Load 3-second features (primary dataset)
data_3s = pd.read_csv('features_3_sec.csv')
data_3s = data_3s.drop(['filename', 'length'], axis=1)
print(f"3-sec dataset: {data_3s.shape}")

# Load 30-second features (supplementary dataset)
data_30s_path = 'features_30_sec.csv'
if os.path.exists(data_30s_path):
    data_30s = pd.read_csv(data_30s_path)
    data_30s = data_30s.drop(['filename', 'length'], axis=1)
    print(f"30-sec dataset: {data_30s.shape}")
    # Combine both datasets
    data = pd.concat([data_3s, data_30s], ignore_index=True)
    print(f"Combined dataset: {data.shape}")
else:
    data = data_3s
    print("30-sec dataset not found, using 3-sec only")

print(f"\nGenres: {list(data['label'].unique())}")
print(f"\nSamples per genre:")
data['label'].value_counts()

3-sec dataset: (9990, 58)
30-sec dataset: (1000, 58)
Combined dataset: (10990, 58)

Genres: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

Samples per genre:


label
blues        1100
jazz         1100
metal        1100
pop          1100
reggae       1100
disco        1099
classical    1098
hiphop       1098
rock         1098
country      1097
Name: count, dtype: int64

## 2. Feature Preparation & Proper 3-Way Split
**Key change:** Using a separate validation set so the test set remains truly unseen.

In [3]:
X = data.drop(['label'], axis=1)
y = data['label']

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Proper 3-way split: 70% train / 15% val / 15% test ---
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.15, random_state=42, stratify=y_encoded
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val
)

print(f"Training samples:   {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples:       {X_test.shape[0]}")
print(f"Features:           {X_train.shape[1]}")

# One-hot encode labels
y_train_cat = to_categorical(y_train)
y_val_cat = to_categorical(y_val)
y_test_cat = to_categorical(y_test)
num_classes = y_train_cat.shape[1]

Training samples:   7696
Validation samples: 1645
Test samples:       1649
Features:           57


## 3. Improved Model Architecture
- **Wider layers:** 512â†’256â†’128â†’64â†’10
- **L2 regularization:** kernel_regularizer=l2(1e-4)
- **Label smoothing:** 0.1 to prevent overconfidence

In [4]:
REG = l2(1e-4)  # L2 regularization strength

model = Sequential([
    # Layer 1 - Wide input layer
    Dense(512, activation='relu', input_shape=(X_train.shape[1],),
          kernel_regularizer=REG),
    BatchNormalization(),
    Dropout(0.4),

    # Layer 2
    Dense(256, activation='relu', kernel_regularizer=REG),
    BatchNormalization(),
    Dropout(0.3),

    # Layer 3
    Dense(128, activation='relu', kernel_regularizer=REG),
    BatchNormalization(),
    Dropout(0.3),

    # Layer 4
    Dense(64, activation='relu', kernel_regularizer=REG),
    BatchNormalization(),
    Dropout(0.2),

    # Output layer
    Dense(num_classes, activation='softmax')
])

# --- Label Smoothing in loss ---
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │        29,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 206,666 (807.29 KB)

 Trainable params: 204,746 (799.79 KB)

 Non-trainable params: 1,920 (7.50 KB)

## 4. Training with EarlyStopping & LR Scheduling
- **EarlyStopping:** patience=20, restores best weights
- **ReduceLROnPlateau:** patience=7, halves LR on plateau
- **ModelCheckpoint:** saves best model to disk

In [5]:
# --- Callbacks ---
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=7,
    min_lr=1e-6,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_ann_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=0
)

# --- Train ---
history = model.fit(
    X_train, y_train_cat,
    epochs=150,
    batch_size=32,
    validation_data=(X_val, y_val_cat),
    callbacks=[lr_scheduler, early_stop, checkpoint],
    verbose=1
)

Epoch 1/150
241/241 ━━━━━━━━━━━━━━━━━━━━ 13s 19ms/step - accuracy: 0.4410 - loss: 1.9040 - val_accuracy: 0.6705 - val_loss: 1.3704 - learning_rate: 0.0010
Epoch 2/150
241/241 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.6051 - loss: 1.4905 - val_accuracy: 0.7489 - val_loss: 1.1760 - learning_rate: 0.0010
Epoch 3/150
241/241 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.6754 - loss: 1.3560 - val_accuracy: 0.7745 - val_loss: 1.1070 - learning_rate: 0.0010
Epoch 4/150
241/241 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.6995 - loss: 1.2845 - val_accuracy: 0.7970 - val_loss: 1.0525 - learning_rate: 0.0010
Epoch 5/150
241/241 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.7218 - loss: 1.2294 - val_accuracy: 0.8140 - val_loss: 1.0342 - learning_rate: 0.0010
Epoch 6/150
241/241 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.7516 - loss: 1.1778 - val_accuracy: 0.8286 - val_loss: 1.0043 - learning_rate: 0.0010
Epoch 7/150
241/241 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.7727 - 

## 5. Evaluation on Held-Out Test Set

In [13]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)

print(f"  TEST ACCURACY:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  TEST LOSS:      {test_loss:.4f}")
print(f"  BEST VAL LOSS:  {min(history.history['val_loss']):.4f}")
print(f"  EPOCHS TRAINED: {len(history.history['loss'])}")


  TEST ACCURACY:  0.9448 (94.48%)
  TEST LOSS:      0.7327
  BEST VAL LOSS:  0.7116
  EPOCHS TRAINED: 150


## 6. Predict Genre from Audio File
Uses Librosa to extract features from raw audio, splits the track into 3-second chunks, and picks the most common prediction as the final genre.

In [12]:
import librosa
from collections import Counter

# --- Feature Extraction ---
def extract_features(y, sr):
    features = []

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features.append(np.mean(chroma))
    features.append(np.var(chroma))

    rms = librosa.feature.rms(y=y)
    features.append(np.mean(rms))
    features.append(np.var(rms))

    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    features.append(np.mean(spec_cent))
    features.append(np.var(spec_cent))

    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features.append(np.mean(spec_bw))
    features.append(np.var(spec_bw))

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features.append(np.mean(rolloff))
    features.append(np.var(rolloff))

    zcr = librosa.feature.zero_crossing_rate(y)
    features.append(np.mean(zcr))
    features.append(np.var(zcr))

    harmony = librosa.effects.harmonic(y)
    features.append(np.mean(harmony))
    features.append(np.var(harmony))

    perceptr = librosa.effects.percussive(y)
    features.append(np.mean(perceptr))
    features.append(np.var(perceptr))

    tempo = librosa.beat.tempo(y=y, sr=sr)[0]
    features.append(tempo)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    for i in range(20):
        features.append(np.mean(mfcc[i]))
        features.append(np.var(mfcc[i]))

    return np.array(features)

# --- Predict Genre ---
def predict_song(file_path):
    y, sr = librosa.load(file_path, duration=30)
    chunk_samples = 3 * sr
    predictions = []

    for i in range(10):
        start = i * chunk_samples
        end = start + chunk_samples
        chunk = y[start:end]

        if len(chunk) < chunk_samples:
            continue

        features = extract_features(chunk, sr)
        features = features.reshape(1, -1)
        features = scaler.transform(features)

        pred = model.predict(features, verbose=0)
        predictions.append(le.inverse_transform([np.argmax(pred)])[0])

    final_genre = Counter(predictions).most_common(1)[0][0]
    print(f"Chunk Predictions: {predictions}")
    print(f"Final Genre: {final_genre}")
    return final_genre

# --- Test on an audio file ---
predict_song("disco22.wav")

Chunk Predictions: ['country', 'country', 'rock', 'rock', 'rock', 'disco', 'disco', 'disco', 'disco', 'disco']
Final Genre: disco


'disco'